# Detection Project — Simple GitHub → Kaggle Save Version (manual approval gated)

**One-click after setup:** keep the *same already-preflighted inputs* used in Chat27 (Split-B + the verified Chat26 historical support datasets). Import this notebook from GitHub, select Kaggle **T4×2**, enable Internet, and **Save Version → Save & Run All**. It automatically clones the exact frozen release, verifies pins, restores R0 paths, builds MV00/MV01 plans, and — *only if you explicitly authorize* — trains two independent workers and exports full G4 checkpoints. No additional Kaggle **authorization dataset** is required.

**Preflight default (SAFE):** `OFFICIAL_USER_CONFIRMATION = ""`. This only runs source/data/governance preflight, with no GPU training.

**To launch officially:** read the run plan and *personally type* the exact value of `REQUIRED_APPROVAL` into `OFFICIAL_USER_CONFIRMATION` in the notebook cell. Your typed approval is saved as a run-local JSON file in `/kaggle/working`, and the immutable C1 approval gate validates it. Nothing is authorized merely by importing or cloning a notebook. Do not type the approval phrase without intending to start both official 100-epoch experiments. Frozen scientific commit: `fa6a7675daae385a04c7b77e69a9de29f9e1b2f3`; test access **NONE**.

**Fresh:** leave `MODE="FRESH"`; session 1 attempts epochs 1–12, stopping earlier if the bounded wall-clock/G4 export safety margin requires. Nominal experiment length stays 100 total epochs. This run remains untested on GPU in official execution until an actual authorized Kaggle version executes.

**Resume:** after each saved version, independently verify each arm's original G4 archive bytes outside the running Kaggle session and attach those attestation JSONs and **one latest successful official saved Kaggle output** as inputs. Set `MODE="RESUME"` and type the approval phrase again. The launcher discovers arm-specific epoch cursors and ZIP hashes; it does not accept just `best.pt`, acceptance-only epochs, or unchecked output bytes. The prior scientific code, training parameters, B-TEST seal, and full-state resume gate remain unchanged.

**Historical support:** Git alone cannot replace original pretrained checkpoint archives and R0 pin evidence. Because the existing Kaggle input set already passed the full Chat27 preflight, keep it attached; do not re-upload/reconfigure it.

In [ ]:
from pathlib import Path
import hashlib, io, json, os, runpy, shutil, subprocess, sys, time, zipfile
from datetime import datetime, timezone

# --- REVIEWED IDENTITY. Never change these without new release review. ---
SHA = "fa6a7675daae385a04c7b77e69a9de29f9e1b2f3"
BASE = "b783eb3a470ab010785a8a24a90a666e9a3f2ed6"
PORTABLE_SHA = "3f0a386ace43453921ddd84b0e5f1aef5dacb72493d11fd5fbadcfad3c974e3c"
RESTAGER_SHA = "88cd4a88c373f451e54ba64736e961e735757d94b3779feea55a526868c45f4f"
# ONE USER ACTION TO START: type this exact sentence into the empty string.
# Leaving it empty makes the entire notebook run PRE-FLIGHT ONLY; no training.
REQUIRED_APPROVAL = "I APPROVE OFFICIAL MV00 MV01 100 EPOCH TRAINING WITH B TEST SEALED"
OFFICIAL_USER_CONFIRMATION = ""
EXECUTE_APPROVED_OFFICIAL = (OFFICIAL_USER_CONFIRMATION == REQUIRED_APPROVAL)
RID = {"MV00":"C1OFFICIAL_MV00_20261022", "MV01":"C1OFFICIAL_MV01_20261022"}

# USER CONFIGURATION: FRESH for session1, RESUME after saved official output is mounted.
MODE = "FRESH"                    # For later sessions, change to "RESUME".
if MODE not in ("FRESH", "RESUME"):
    raise RuntimeError("MODE_MUST_BE_FRESH_OR_RESUME")
if MODE=="FRESH":
    ARMS={arm:{"mode":"FRESH","start":1,"target":12,"resume_sha256":None} for arm in RID}
    SESSION_LABEL="FRESH01"
else:
    # Kaggle Add Input: attach exactly one latest saved OFFICIAL session output.
    matches=[]
    for f in Path("/kaggle/input").rglob("C1_OUTPUT_INDEX.json"):
        if not f.is_file() or f.is_symlink():continue
        try: info=json.loads(f.read_text("utf-8"))
        except (OSError,ValueError,UnicodeError):continue
        if (info.get("run_class")=="OFFICIAL_100EPOCH" and
            info.get("published_release_commit")==SHA and
            info.get("both_arms_completed") is True and
            info.get("status")=="BOTH_ARMS_LOCAL_OUTPUTS_VERIFIED_EXTERNAL_PENDING" and
            info.get("offsite_persistence_verified") is False and
            info.get("b_test_access")=="NONE" and not info.get("failures") and
            set(info.get("arms",{}))==set(RID) and
            all(info["arms"][x].get("run_id")==RID[x] for x in RID)):
            matches.append((f,info))
    if len(matches)!=1:
        raise RuntimeError("RESUME_REQUIRES_ONE_LATEST_SAVED_OFFICIAL_OUTPUT_INDEX")
    previous_output_index,previous_record=matches[0]
    ARMS={}
    for arm in RID:
        prev=previous_record["arms"][arm]
        cursor,pin=prev.get("completed_epoch_zero_based"),prev.get("zip_sha256")
        if (type(cursor) is not int or not 0<=cursor<99 or
            prev.get("terminal") is not False or
            type(pin) is not str or len(pin)!=64 or
            any(c not in "0123456789abcdef" for c in pin)):
            raise RuntimeError("PRIOR_ARCHIVE_INVALID_OR_TERMINAL_"+arm)
        nxt=cursor+2
        ARMS[arm]={"mode":"RESUME","start":nxt,"target":min(nxt+11,100),
                   "resume_sha256":pin}
    SESSION_LABEL="R"+str(max(ARMS[x]["start"] for x in RID)).zfill(3)
print("GOVERNED_SESSION",MODE,SESSION_LABEL,ARMS,flush=True)
SESSION_SECONDS = 32400     # 9-hour native-worker wall-clock deadline
MINIMUM_EPOCH_SECONDS = 1800
EXPORT_RESERVE_SECONDS = 3600  # > C1 required 30-minute minimum
ROOT = Path("/kaggle/input")
REPO = Path("/kaggle/working/c1_official_fixed_checkout")
PLAN = Path("/kaggle/working/CHAT27_C1_OFFICIAL_PLANS_" + SESSION_LABEL)
OUT  = Path("/kaggle/working/CHAT27_C1_OFFICIAL_OUTPUT_" + SESSION_LABEL)
LOG  = Path("/kaggle/working/CHAT27_C1_OFFICIAL_LOGS_" + SESSION_LABEL)

def need(ok, reason):
    if not ok: raise RuntimeError("CHAT27_OFFICIAL_PRECHECK_FAIL_CLOSED_" + reason)

def digest(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for b in iter(lambda:f.read(1<<20),b""): h.update(b)
    return h.hexdigest()

need(ROOT.is_dir(), "KAGGLE_INPUTS_MISSING")
need(SESSION_LABEL.isalnum() and 1<=len(SESSION_LABEL)<=16, "SESSION_LABEL")
need(set(ARMS)==set(RID) and RID["MV00"]!=RID["MV01"], "TWO_INDEPENDENT_ARMS")
for arm, a in ARMS.items():
    need(set(a)=={"mode","start","target","resume_sha256"}, "ARM_KEYS_"+arm)
    need(a["mode"] in ("FRESH","RESUME"), "MODE_"+arm)
    st, tar = a["start"],a["target"]
    need(type(st) is int and type(tar) is int and 1<=st<=tar<=100 and tar-st+1<=12,
         "SESSION_BOUNDS_"+arm)
    if a["mode"]=="FRESH":
        need(st==1 and a["resume_sha256"] is None, "FRESH_MUST_RESTART_EPOCH1_"+arm)
    else:
        v=a["resume_sha256"]
        need(st>=2 and type(v) is str and len(v)==64 and
             all(c in "0123456789abcdef" for c in v), "REQUIRE_INDEPENDENT_G4_SHA_"+arm)
need(not any(x.exists() or x.is_symlink() for x in (REPO,PLAN,OUT,LOG)),
     "SESSION_MUST_START_CLEAN")
need(type(EXECUTE_APPROVED_OFFICIAL) is bool and
     0<SESSION_SECONDS<=36000 and EXPORT_RESERVE_SECONDS>=1800, "SESSION_BUDGET")
gpu=subprocess.run(["nvidia-smi","-L"],capture_output=True,text=True)
gpu_lines=[v for v in gpu.stdout.splitlines() if v.startswith("GPU ")]
need(gpu.returncode==0 and len(gpu_lines)==2 and
     all("T4" in x.upper() for x in gpu_lines), "TWO_PHYSICAL_T4_REQUIRED")
print("T4_X2_AND_SESSION_CONFIG_PASS",SESSION_LABEL,flush=True)
print("OFFICIAL_EXECUTION_FLAG",EXECUTE_APPROVED_OFFICIAL, "B_TEST_NONE", flush=True)


# -------- Automatic next stage --------

# Exact Git checkout; R8 verifies additive-only release ancestry.
subprocess.run(["git","clone","--no-checkout",
                "https://github.com/nithoukhan1/ResEMA.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",SHA],check=True)
need(subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],
                            text=True).strip()==SHA, "CHECKOUT_SHA")
need(REPO.is_dir() and not REPO.is_symlink() and
     not any(p.is_symlink() for p in (REPO,REPO.parent)), "CHECKOUT_SYMLINK")
REL=REPO/"research/r9_training"
C1=REL/"official_candidate"
CODE=REL/"code"
sys.path[:0]=[str(C1),str(CODE)]
from R9_OFFICIAL_RELEASE_IDENTITY_R8 import validate_checkout
identity=validate_checkout(REPO,SHA)
need(identity["status"]=="R8_EXACT_CHECKOUT_SHA_AND_ADDITIVE_DIFF_VERIFIED_NOT_REMOTE_PROOF",
     "RELEASE_IDENTITY")
print("EXACT_FROZEN_C1_RELEASE_PASS",SHA,flush=True)


# -------- Automatic next stage --------

# Restore the 16 historical working paths with the exact Chat25 corrected restager.
NAME="CHAT25_C1_R0_STALE_PATH_SAFE_RESTAGE_CORRECTED.py"
def member_safe(s):
    from pathlib import PurePosixPath
    p=PurePosixPath(s)
    return not p.is_absolute() and ".." not in p.parts and "\\" not in s
def find_inside_zip(blob,depth=0):
    if depth>3:return []
    found=[]
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for m in z.infolist():
            if m.is_dir() or not member_safe(m.filename):continue
            if (m.filename==NAME or m.filename.endswith("/"+NAME)):
                need(m.file_size<2_000_000, "RESTAGER_SIZE")
                found.append(z.read(m))
            elif m.filename.lower().endswith(".zip") and m.file_size<120_000_000:
                found.extend(find_inside_zip(z.read(m),depth+1))
    return found
restagers=[p.read_bytes() for p in ROOT.rglob(NAME) if p.is_file() and not p.is_symlink()]
if not restagers:
    for p in ROOT.rglob("*"):
        if (p.is_file() and not p.is_symlink() and
            "DETECTION_PROJECT_MASTER_MIGRATION_CHAT1_TO_CHAT25_FOR_CHAT26_2026-10-21_PORTABLE" in p.name and
            p.suffix.lower() in (".zip",".zipblob")):
            need(digest(p)==PORTABLE_SHA, "CHAT25_PORTABLE_WRONG_BYTES")
            restagers.extend(find_inside_zip(p.read_bytes()))
need(bool(restagers), "ORIGINAL_CORRECTED_R0_RESTAGER_NOT_MOUNTED")
need(len({hashlib.sha256(v).hexdigest() for v in restagers})==1,
     "CONFLICTING_RESTAGER_COPIES")
script=Path("/kaggle/working")/NAME
with script.open("xb") as f:f.write(restagers[0])
need(digest(script)==RESTAGER_SHA,"CORRECTED_R0_SHA_MISMATCH")
expected_config=Path("/kaggle/input/datasets/nithoukhan2/detect-1/R7D_RESTORE_CONTEXT_SUPPORT_R1/inputs/RECOVERY_CONFIG.json")
expected_sci=REL/"frozen_source/ultralytics/models/yolo/detect/d4e_r3_read_only_identity_preflight.py"
need(expected_config.is_file() and not expected_config.is_symlink() and
     expected_sci.is_file() and not expected_sci.is_symlink(),
     "R7D_CONFIG_OR_FROZEN_SCIENCE_NOT_MOUNTED")
runpy.run_path(str(script),run_name="__main__")
print("EXACT_CHAT25_R0_RESTAGE_EXECUTED",flush=True)


# -------- Automatic next stage --------

# Identify the exact original six inputs by SHA, never by convenient filename alone.
PINS={
 "runtime_yaml":("f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",(".yaml",".yml")),
 "assignment":("87aa5e9089c6282df14ae32917946d70d9f56eaa9a5d63443985877ab9dd1ef4",(".csv",".json")),
 "visibility":("9de034fa7df668d17effa843674c097341af9dd6461963f9e2c454c5f358b289",(".json",)),
 "pretrained_best":("620594d3560310b5d24243e1b321ce4463b0795b295891c47f6ec177e5272ac1",(".pt",)),
 "pretrained_archive":("4e3a811be5b8126e788ca4b7854e2df2b070a08e2ab8f0c6bbde8b6e1abef396",(".bin",".zip",".zipblob"))
}
MOUNTED=("d4e-r12-accepted-assets","d4e-r12-raw-archives-v1",
         "detect-1","detection","d4f-r7d-legacy-exact-source-evidence-r1")
files=[]
for dirpath, dirs, fnames in os.walk(ROOT):
    dirs[:]=[d for d in dirs if not d.startswith(".")]
    for name in fnames:
        p=Path(dirpath)/name
        if any(s in str(p) for s in MOUNTED) and p.is_file() and not p.is_symlink():
            files.append(p)
INPUT={}
for key, (pin,suffixes) in PINS.items():
    matches=[p for p in files if p.name.lower().endswith(suffixes) and digest(p)==pin]
    need(bool(matches),"MISSING_EXACT_PIN_"+key)
    INPUT[key]=str(sorted(matches,key=str)[0])
configs=[p for p in files if p.name=="RECOVERY_CONFIG.json" and
         "R7D_RESTORE_CONTEXT_SUPPORT_R1/inputs" in str(p)]
need(len(configs)==1 and configs[0]==expected_config, "ORIGINAL_R7D_CONFIG")
INPUT["config_path"]=str(configs[0])
print("ORIGINAL_SIX_INPUT_IDENTITIES_PASS",json.dumps(INPUT,indent=2),flush=True)


# -------- Automatic next stage --------

# Only verified, genuinely offsite official per-arm G4 can enable RESUME.
from R9_OFFSESSION_TRANSPORT_R1 import inspected_archive
def prior_triplet(arm, settings):
    if settings["mode"]=="FRESH":return (None,None,None)
    ep=settings["start"]-1
    pin=settings["resume_sha256"]
    record=previous_record["arms"][arm]
    need(record["completed_epoch_zero_based"]==ep-1 and record["zip_sha256"]==pin,
         "SAVED_INDEX_CURSOR_OR_SHA_"+arm)
    saved=previous_output_index.parent/record["zip_relative"]
    blobs=[saved] if saved.is_file() and not saved.is_symlink() and digest(saved)==pin else []
    need(len(blobs)==1,"REQUIRE_ONE_EXACT_VERIFIED_OFFSITE_G4_"+arm)
    blob=blobs[0]
    # C1 stage_available retains ORIGINAL sidecar name: original.zip.sha256.
    # The staged ZIP is renamed original.zip.zipblob, so appending '.sha256'
    # to the staged ZIP name would point to a nonexistent/different file.
    need("sidecar_relative" in record and "sidecar_sha256" in record,
         "SAVED_INDEX_SIDECAR_IDENTITY_"+arm)
    side=previous_output_index.parent/record["sidecar_relative"]
    original_name=blob.name.removesuffix(".zipblob")
    need(blob.name.endswith(".zip.zipblob") and
         side.name==original_name+".sha256" and
         side.is_file() and not side.is_symlink() and
         digest(side)==record["sidecar_sha256"] and
         side.read_text("utf-8").strip().split()==[pin,original_name],
         "ARCHIVE_ORIGINAL_SIDE_CAR_"+arm)
    checked=inspected_archive(blob,sidecar=side)
    need(checked["status"]=="PASS_BYTE_LEVEL_ARCHIVE_VERIFICATION_NOT_GPU_REPRODUCTION"
         and checked["arm"]==arm and checked["completed_epoch"]==ep-1
         and checked["terminal"] is False and checked["zip_sha256"]==pin,
         "ARCHIVE_NOT_MATCHING_CURSOR_"+arm)
    found=[]
    for f in ROOT.rglob("*.json"):
        if not f.is_file() or f.is_symlink():continue
        try: j=json.loads(f.read_text("utf-8"))
        except (ValueError, OSError, UnicodeError):continue
        if (j.get("schema")=="R9_OFFSITE_CHECK_NOT_NATIVE_RESUME_ACCEPTANCE" and
            j.get("arm")==arm and j.get("completed_epoch")==ep-1 and
            j.get("terminal") is False and j.get("zip_sha256")==pin):
            found.append(f)
    need(len(found)==1, "REQUIRE_INDEPENDENT_OFFSESSION_RECEIPT_"+arm)
    return (str(blob),str(side),str(found[0]))
RESUME={arm:prior_triplet(arm,ARMS[arm]) for arm in RID}
print("PER_ARM_OFFSITE_RESUME_INPUTS",{
    a:("FRESH" if RESUME[a][0] is None else "OFFSITE_G4_BYTES_VERIFIED") for a in RID},
    flush=True)


# -------- Automatic next stage --------

# Source/data/R0 pin verification and official 2-arm session plan; still no GPU.
from R9_PRODUCTION_PLAN_FACTORY_C1 import create
from R9_PRODUCTION_APPROVAL_GATE_C1 import validate as verify_approval
requests={}
for arm in RID:
    a=ARMS[arm]
    triple=RESUME[arm]
    requests[arm]={"mode":"FRESH_FROM_EARLY" if a["mode"]=="FRESH" else "RESUME_FROM_OUR_OWN_RUN",
                   "start_epoch_one_based":a["start"],"target_epoch_one_based":a["target"],
                   "resume_archive":triple[0],"resume_sidecar":triple[1],
                   "resume_external_attestation":triple[2]}
request={"run_ids":RID, "arms":requests, "monotonic_session_seconds":SESSION_SECONDS,
         "minimum_epoch_seconds":MINIMUM_EPOCH_SECONDS,
         "reserve_export_seconds":EXPORT_RESERVE_SECONDS}
result=create(INPUT,request,checkout=str(REPO),release_sha=SHA,output_dir=str(PLAN))
need(result["status"]=="TWO_ARM_PINNED_PLANS_CREATED_NO_GPU","PLAN_FACTORY")
specs={arm:json.loads((PLAN/(arm+".json")).read_text()) for arm in RID}
review=verify_approval(specs,None,SHA,execute=False)
need(review["official_training_authorized"] is False, "NO_IMPLICIT_AUTH")
print("TWO_ARM_OFFICIAL_PLAN_REVIEW_PASS_NO_GPU",json.dumps(result,indent=2),flush=True)


# -------- Automatic next stage --------

# The *user* must type the exact affirmative sentence above. The code ships
# with an EMPTY confirmation and NO authorized record. When explicitly confirmed,
# this Kaggle session writes the C1-required approval JSON to /kaggle/working.
# The immutable C1 approval gate still validates every field before GPU launch.
APPROVAL_NAME="CHAT27_C1_OFFICIAL_USER_AUTHORIZATION.json"
if not EXECUTE_APPROVED_OFFICIAL:
    print("REVIEW_ONLY_NO_GPU: exact release/source/data/plans validated.")
    print("No training launched. To authorize, enter REQUIRED_APPROVAL exactly in OFFICIAL_USER_CONFIRMATION.")
else:
    need(OFFICIAL_USER_CONFIRMATION == REQUIRED_APPROVAL,
         "EXPLICIT_USER_APPROVAL_SENTENCE_REQUIRED")
    AUTH=Path("/kaggle/working")/APPROVAL_NAME
    need(not AUTH.exists() and not AUTH.is_symlink(),"NEW_APPROVAL_RECORD_ONLY")
    approval={
        "schema": "DETECTION_CHAT25_OFFICIAL_FRESH_100EPOCH_RUN_AUTHORIZATION_C1",
        "purpose": "USER_APPROVED_FINAL_PRODUCTION_RELEASE_AND_EXACT_FRESH_100EPOCH_MV00_MV01",
        "approval_record": "EXPLICIT_FINAL_USER_APPROVAL_RECORDED",
        "release_commit": SHA,
        "frozen_science_commit": BASE,
        "run_ids": RID.copy(),
        "official_training_authorized": True,
        "b_test_access": "NONE",
        "max_epochs": 100,
        "checkpoint_retention": "KEEP_ORIGINAL_UNTIL_EXTERNAL_SHA_VERIFIED",
        "fresh_pretrained_only": True,
    }
    with AUTH.open("x",encoding="utf-8") as f:
        json.dump(approval,f,indent=2,sort_keys=True)
        f.write("\n")
        f.flush()
        os.fsync(f.fileno())
    auth=verify_approval(specs,approval,SHA,execute=True)
    need(auth["status"]=="OFFICIAL_RUN_APPROVAL_RECORD_MATCHED_NOT_EXECUTION_PROOF",
         "EXACT_USER_AUTHORITY_AND_RUN_IDENTITIES")
    print("EXPLICIT_KAGGLE_USER_APPROVAL_RECORD_VALIDATED",str(AUTH),flush=True)
    import torch
    need(torch.cuda.is_available() and torch.cuda.device_count()==2 and
         all("T4" in torch.cuda.get_device_name(i).upper() for i in (0,1)),
         "CUDA_T4_X2_REQUIRED")
    need(not OUT.exists() and not LOG.exists(),"FRESH_OUTPUT_LOG_ROOTS_REQUIRED")
    need(all(not Path(specs[a]["working_root"]).exists() for a in RID),
         "WORKER_OUTPUT_MUST_BE_FRESH")
    cmd=[sys.executable,"-u",str(C1/"R9_PRODUCTION_DUAL_SESSION_C1.py"),
         "--mv00",str(PLAN/"MV00.json"),"--mv01",str(PLAN/"MV01.json"),
         "--release-sha",SHA,"--authorization",str(AUTH),
         "--output-root",str(OUT),"--logs-root",str(LOG),
         "--execute-approved-final-run"]
    supervisor=Path("/kaggle/working")/("CHAT27_C1_"+SESSION_LABEL+"_SUPERVISOR.log")
    incident=Path("/kaggle/working")/("CHAT27_C1_"+SESSION_LABEL+"_INCIDENT.zip")
    need(not supervisor.exists() and not incident.exists(), "ORIGINAL_LOGS_MUST_BE_FRESH")

    def tail(path):
        if not path.is_file():return "not-yet-created"
        with path.open("rb") as f:
            f.seek(max(0,path.stat().st_size-8192))
            b=f.read().decode("utf-8",errors="replace")
        a=[v.strip() for v in b.replace("\r","\n").splitlines() if v.strip()]
        return a[-1][:230] if a else "empty"

    def incident_archive():
        if incident.exists():return
        picks=[PLAN/"MV00.json",PLAN/"MV01.json",PLAN/"PLAN_REVIEW.json",supervisor,
               LOG/"MV00_native_stdout_stderr.log",LOG/"MV01_native_stdout_stderr.log",
               OUT/"C1_OUTPUT_INDEX.json"]
        with zipfile.ZipFile(incident,"x",zipfile.ZIP_DEFLATED) as z:
            for p in picks:
                if p.is_file() and not p.is_symlink():
                    z.write(p,p.relative_to("/kaggle/working").as_posix())
        Path(str(incident)+".sha256").write_text(
            digest(incident)+"  "+incident.name+"\n",encoding="utf-8")
        print("PRESERVED_NONCANONICAL_INCIDENT",str(incident),flush=True)

    print("OFFICIAL_DUAL_T4_SESSION_START",SESSION_LABEL,flush=True)
    print("TRAINING_LABEL_SCAN_CAN_TAKE_MANY_MINUTES; NO INTERRUPT_OR_RERUN",flush=True)
    with supervisor.open("x",encoding="utf-8") as sink:
        proc=subprocess.Popen(cmd,stdout=sink,stderr=subprocess.STDOUT,
                              stdin=subprocess.DEVNULL)
        try:
            while proc.poll() is None:
                print("UTC_HEARTBEAT",datetime.now(timezone.utc).isoformat(),flush=True)
                for a in RID:
                    print(a,tail(LOG/(a+"_native_stdout_stderr.log")),flush=True)
                time.sleep(75)
        except KeyboardInterrupt:
            print("INTERRUPTED_DO_NOT_RESTART_IN_THIS_KAGGLE_SESSION",flush=True)
            if proc.poll() is None:
                proc.terminate()
                try:proc.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    proc.kill();proc.wait()
            incident_archive()
            raise
        rc=proc.wait()
    print("OFFICIAL_DUAL_SUPERVISOR_EXIT",rc,flush=True)
    if rc!=0:
        incident_archive()
        print("SUPERVISOR_TAIL",tail(supervisor),flush=True)
        for a in RID:
            print("ARM_LOG_TAIL",a,tail(LOG/(a+"_native_stdout_stderr.log")),flush=True)
        raise RuntimeError("OFFICIAL_SESSION_FAILED_KEEP_ALL_ORIGINAL_ARTIFACTS_NO_AUTO_RETRY")
    index_path=OUT/"C1_OUTPUT_INDEX.json"
    if not index_path.is_file():
        incident_archive()
        raise RuntimeError("OFFICIAL_OUTPUT_INDEX_MISSING")
    idx=json.loads(index_path.read_text("utf-8"))
    if not (idx.get("both_arms_completed") is True and
            idx.get("status")=="BOTH_ARMS_LOCAL_OUTPUTS_VERIFIED_EXTERNAL_PENDING" and
            idx.get("run_class")=="OFFICIAL_100EPOCH" and
            idx.get("published_release_commit")==SHA and
            idx.get("b_test_access")=="NONE" and
            idx.get("offsite_persistence_verified") is False and
            not idx.get("failures") and set(idx.get("arms",{}))==set(RID)):
        incident_archive()
        raise RuntimeError("OFFICIAL_DUAL_ARM_OUTPUT_STATUS_INVALID")
    summary={}
    for arm, e in idx["arms"].items():
        cursor=e["completed_epoch_zero_based"]
        need(e["run_id"]==RID[arm] and
             ARMS[arm]["start"]-1<=cursor<ARMS[arm]["target"] and
             e["terminal"] is (cursor==99), "ARM_CURSOR_OR_TERMINAL_"+arm)
        blob=OUT/e["zip_relative"]
        need(blob.is_file() and blob.name.endswith(".zipblob") and
             digest(blob)==e["zip_sha256"], "OUTPUT_ZIPBLOB_HASH_"+arm)
        # Verify existing ORIGINAL archive sidecar. Do not create an invented
        # renamed sidecar; its SHA and original .zip name are release evidence.
        need("sidecar_relative" in e and "sidecar_sha256" in e,
             "INDEX_SIDECAR_FIELDS_"+arm)
        side=OUT/e["sidecar_relative"]
        original_name=blob.name.removesuffix(".zipblob")
        need(blob.name.endswith(".zip.zipblob") and
             side.name==original_name+".sha256" and
             side.is_file() and not side.is_symlink() and
             digest(side)==e["sidecar_sha256"] and
             side.read_text("utf-8").strip().split()==[e["zip_sha256"],original_name],
             "STAGED_ORIGINAL_SIDECAR_"+arm)
        worker_log=LOG/(arm+"_native_stdout_stderr.log")
        if worker_log.is_file():
            shutil.copy2(worker_log,OUT/(arm+"_native_stdout_stderr.log"))
        summary[arm]={"completed_epoch_one_based":cursor+1,
                      "terminal":e["terminal"],"g4_sha256":e["zip_sha256"],
                      "next_epoch_one_based":None if cursor==99 else cursor+2}
    shutil.copy2(supervisor,OUT/("CHAT27_C1_"+SESSION_LABEL+"_SUPERVISOR.log"))
    print("OFFICIAL_SESSION_LOCAL_G4_PASS_EXTERNAL_CHECK_REQUIRED",
          json.dumps(summary,indent=2),flush=True)
    print("DO_NOT_RESUME_UNTIL_INDEPENDENT_WINDOWS_OFFSITE_CHECK_PASSES",flush=True)


### Required output preservation

Save Version persists `C1_OUTPUT_INDEX.json`, original staged per-arm `.zip.zipblob` files, their original `.zip.sha256` sidecars, and logs. Before any RESUME, carry out an independently verified off-session transport check, retain originals until checks pass, and supply corresponding arm-specific receipts. The frozen native G4 recovery stage verifies that the optimizer/EMA/scaler continuation is genuine. Neither passing static checks nor producing a Kaggle save proves official epoch100 completion.